## 1、什么是输出解析器：output parser
将大模型的原始自然语言类型的输出，解析成程序所需要的结构化的输出

In [1]:
from langchain_openai import ChatOpenAI
import dotenv
# from llama_index.core import ChatPromptTemplate

dotenv.load_dotenv()
openai_llm = ChatOpenAI(
    model="gpt-4o-mini"
)

C:\Users\m1881\miniconda3\envs\LangChainProj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\m1881\miniconda3\envs\LangChainProj\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [2]:
openai_llm.invoke("帮我推荐几部诺兰的电影")

AIMessage(content='克里斯托弗·诺兰是一位备受推崇的导演，他的许多电影都以复杂的叙事结构和深刻的主题而闻名。以下是几部值得推荐的诺兰电影：\n\n1. **《盗梦空间》（Inception, 2010）** - 一部关于梦境和现实交织的科幻片，讲述了一组专业盗贼如何潜入他人梦中进行意识形态的盗取。\n\n2. **《黑暗骑士三部曲》（The Dark Knight Trilogy）** - 这三部电影包括《蝙蝠侠：侠影之谜》（Batman Begins, 2005）、《黑暗骑士》（The Dark Knight, 2008）和《黑暗骑士崛起》（The Dark Knight Rises, 2012），重新定义了超级英雄电影。\n\n3. **《记忆碎片》（Memento, 2000）** - 一部非线性叙事的惊悚片，讲述一位失去短期记忆的男子如何试图找到杀死他妻子的凶手。\n\n4. **《星际穿越》（Interstellar, 2014）** - 一部关于太空探索和人类生存的科幻片，探讨了爱、时间和科学之间的关系。\n\n5. **《敦刻尔克》（Dunkirk, 2017）** - 以第二次世界大战为背景，讲述了敦刻尔克撤退的真实事件，采用多条时间线和不同视角展现战争的紧张氛围。\n\n6. **《信条》（Tenet, 2020）** - 一部复杂的时间操控科幻片，围绕着逆转时间的概念展开，结合了间谍惊悚和动作元素。\n\n这些电影都展示了诺兰独特的导演风格和对叙事的创新处理，值得一看！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 429, 'prompt_tokens': 16, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_t

## 2、langchain当中的OutputParser
### JsonOutputParser

#### 1、通过pydantic类去定义JSON的结构，并且构造JsonOutputParser的实例

In [64]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel,Field
# 1、构造实例
#构造参数： pydantic_object: pydantic类
# pydantic包的作用：定义一个数据结构（schema）
# JsonOutputParser使用pydantic定义的数据结构，来定义json当中key有哪些，分别的结构是什么
class FilmSuggestion(BaseModel):
    film_name :str = Field(description="电影的名称")
    year: str = Field(description="上映年份")
    descripion :str =Field(description="电影的梗概")

In [67]:
# 使用pydantic模型的作用举例：
# 构造一个按照BaseModel类定义的字段，以及传入正确的类型
film_suggestion = FilmSuggestion(film_name="盗梦空间",year="2025",descripion="一部电影")
print(film_suggestion)
# 输入错误的类型，pydantic的验证机制会触发并报错
film_suggestion = FilmSuggestion(film_name="盗梦空间",year=2025,descripion="一部电影")
film_suggestion

film_name='盗梦空间' year='2025' descripion='一部电影'


ValidationError: 1 validation error for FilmSuggestion
year
  Input should be a valid string [type=string_type, input_value=2025, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [69]:
# 通过pydantic model，构造一个json的output_parser实例
output_parser = JsonOutputParser(pydantic_object=FilmSuggestion)

In [70]:
# 调用output_parser.get_format_instructions()，能够输出让大模型按照指定json形式输出的prompt
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


#### 2、具体调用时：
在system_message当中添加上需要大模型按照json结构回复的prompt。添加方式：通过调用output_parser.get_format_instructions。接下来，大模型就能够回复一个指定结构的json字符串

In [56]:
from langchain_core.prompts import ChatPromptTemplate
messages = [
    ("system","回答用户问题，按照以下形式做输出：{output_format}"),
    ("user","{user_question}")
]
template = ChatPromptTemplate.from_messages(
    messages
)
llm_output = openai_llm.invoke(template.invoke({"output_format":output_parser.get_format_instructions(),"user_question":"帮我推荐几部诺兰的电影"}))

In [9]:
print(llm_output.content)
print(type(llm_output.content))

{"film_name":"盗梦空间","year":"2010","descripion":"一名技术娴熟的盗贼能够进入他人的梦境中窃取机密信息，并得到一个反向使命：在目标心中植入一个想法。"}
<class 'str'>


通过调用output_parser.invoke方法，可以更进一步将大模型输出的字符串解析成字典对象

In [10]:
parsered_output = output_parser.invoke(llm_output)
print(parsered_output)
print(type(parsered_output))

{'film_name': '盗梦空间', 'year': '2010', 'descripion': '一名技术娴熟的盗贼能够进入他人的梦境中窃取机密信息，并得到一个反向使命：在目标心中植入一个想法。'}
<class 'dict'>


In [60]:
# 通过chain调用JsonOutPutParser，可以一步到位，获取到字典类型的对象
dict_chain = template | openai_llm | output_parser

parsered_output = dict_chain.invoke({"output_format":output_parser.get_format_instructions(),"user_question":"帮我多推荐几部诺兰的电影，大于等于1部"})

In [12]:
parsered_output["year"]

'2010'

### StrOutPutParser
作用：从结果中提取出content字段的内容

In [28]:
# 1、构造实例
from langchain_core.output_parsers import StrOutputParser
str_output_parser = StrOutputParser()

In [32]:
llm_output = openai_llm.invoke("你是谁")

In [35]:
# 当前llm_output 是一个AIMessage的一个实例，在实例当中，封装了大模型的输出结果
# 通过调用str_output_parser.invoke方法，就能够解析获取到大模型输出的content结果
str_output_parser.invoke(llm_output)

'我是一个人工智能助手，旨在回答你的问题和提供帮助。有什么我可以为你做的吗？'

In [37]:
# 在链式调用当中，能够一步到位，获取到字符串类型的大模型输出结果
chain = openai_llm | str_output_parser

In [38]:
chain.invoke("你是谁")

'我是一个人工智能助手，旨在回答问题、提供信息和帮助解决问题。你有什么想了解的呢？'

## 大模型自身提供的Structured Output

In [43]:
# # 1、通过llm对象，调用with_structured_output方法，返回一个新的llm的实例
# 给openai_llm传递一个schema参数即可，参数值为pydantic的类
structured_llm = openai_llm.with_structured_output(schema=FilmSuggestion)
# from langchain_anthropic import ChatAnthropic

In [44]:
# 新的llm实例，在调用invoke方法，返回的结果，就是一个结构化的对象
structured_llm.invoke("帮我推荐几部诺兰的电影")

FilmSuggestion(film_name='盗梦空间', year='2010', descripion='一名专业的梦境盗贼被邀请执行一项前所未有的任务：在目标的潜意识中植入一个想法。他与团队合作，进入多重梦境，挑战现实与幻想的界限。')